## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
import keras
import numpy.matlib
import mat73
import colorcet as cc

from numpy import asarray
from scipy import stats, signal, io
from scipy.ndimage import median_filter as medfilt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.utils.extmath import randomized_svd as rsvd

from utils import tic, toc, hankel_matrix, scaler, legendre_polys, find_opt_lag
from models import linear_regression, VAE

%matplotlib notebook

## Load and preprocess data

In [2]:
def load_pupil(subj,filt=True):
    data_dict = mat73.loadmat(data_dir + '/' + str(subj) + '_pupil.mat')
    pupil = data_dict['pupil'].reshape(-1,1)
    pupil = medfilt(pupil,21)
    
    if filt:
        Fs = 20
        dt = 1/Fs
        sos = signal.butter(1, [.01,.2],btype = 'bandpass', output = 'sos', fs = Fs)
        pupil = signal.sosfiltfilt(sos, pupil, axis=0)
    
    pupil = pupil[1000:-800,:]
    
    pupil = StandardScaler(with_std=False).fit_transform(pupil)

    return pupil

In [3]:
def load_brain(subj,modality='jrgeco',filt=True):
    data_dict = mat73.loadmat(data_dir + '/' + str(subj) + '_' + modality + '.mat')
    brain = data_dict['data'][:,mask_long]

    if filt:
        Fs = 20
        dt = 1/Fs
        sos = signal.butter(1, [.01,.2],btype = 'bandpass', output = 'sos', fs = Fs)
        brain = signal.sosfiltfilt(sos, brain, axis=0)
        
    brain = brain[1000:-800,:]
    
    brain = StandardScaler(with_std=False).fit_transform(brain)

    return brain

In [4]:
data_dir = '/home/andrew/synology/andrew/arousal_dynamics/data'
outdir = '/home/andrew/synology/andrew/arousal_dynamics/model_predictions'
mask = io.loadmat(data_dir+'/newmask.mat')
mask = mask["newmask"]
mask.dtype = bool
mask_long = mask.reshape(16384,order='F')
allen_ccf = mat73.loadmat(data_dir + '/allen_ccf_overlay.mat')['allen_ccf']

In [5]:
from numpy.random import seed
seed(1)
keras.utils.set_random_seed(1)

## Analysis

In [6]:
## Set modeling parameters

subjects = [4,5,6,7,8,9,10]
delay = 1 # implement delay embedding? 0 or 1
nonlinear = 1 # use nonlinear mappings (VAE) (as opposed to linear regression) ? 0 or 1
Fs = 20 # sampling frequency
dt = 1/Fs
stackmax = 100 # Hankel row dimension (i.e., number of time shifts)
spacing = 3 # number of samples separating each row
r = 10 # number of modes of the Hankel matrix to keep (used for projection onto top r Legendre polynomials)
z_n = 4 # number of latent dimensions used for VAE
num_epochs = 200 # number of training epochs
lr = .001 # learning rate
beta = .1 # beta weight for KL loss in VAE training
anneal_step = 2000 # determines rate at which KL loss is increased during training
pretrained_enc = [] # pretrained encoder

In [ ]:
## Create group files

xtrains = []
pupils = []
ytrains = []
yscales = []
lags = []

modality='jrgeco'

for s in range(len(subjects)):
    
    tic()
    
    subj = subjects[s]
    
    # Load data
    pupil = load_pupil(subj)
    brain = load_brain(subj,modality)
    
    # Compute lags for later
    lag_range = 5 # range over which to look for optimal lag between pupil and brain cross-correlation (in seconds)
    lag_range = int(lag_range/dt)
    lag,_,_ = find_opt_lag(pupil,np.mean(brain,axis=1),lag_range)
    lags.append(lag)
          
    ## Time delay embedding
    polys = legendre_polys(r, stackmax)
    Hp = hankel_matrix(pupil.T,stackmax,spacing)
    xtrain = Hp.T@polys
    xtrain = np.concatenate((xtrain,Hp.T[:,-1:]),axis=1)
    ytrain = brain[spacing*(stackmax-1):,:]
        
    xtrain,_ = scaler(xtrain)
    ytrain, scale_y = scaler(ytrain)
    
    xtrains.append(xtrain)
    ytrains.append(ytrain)
    yscales.append(scale_y)
    
    toc()


In [8]:
np.save(outdir + '/group_' + modality + '_lags.npy', lags)
#np.save(outdir + '/group_' + modality + '_ytrains.npy', ytrains, allow_pickle=True)
#np.save(outdir + '/group_xtrains.npy', xtrains)

In [ ]:
# Leave-one-out crossval

num_PCs = 10
modality = 'jrgeco'

#ytrains = np.load(outdir + '/group_' + modality + '_ytrains.npy',allow_pickle=True)
#lags = np.load(outdir + '/group_' + modality + '_lags.npy').tolist()

for s in range(len(xtrains)):
    
    tic()

    print(f'Fold {s+1}')
    
    subj = subjects[s]
        
    for m in range(4):
        if m==0:
            delay = 1
            nonlinear = 1
        elif m==1:
            delay = 1
            nonlinear = 0
        elif m==2:
            delay = 0
            nonlinear = 1
        else:
            delay = 0
            nonlinear = 0
           
        # Apply the median lag adjustment to all mice
        lag = int(np.median(lags[:s] + lags[s+1:]))
        
        xtrains_aligned = xtrains.copy()
        ytrains_aligned = ytrains.copy()
        
        for ss in range(len(xtrains)):
            xtrains_aligned[ss] = xtrains_aligned[ss][lag:,:]
            ytrains_aligned[ss] = ytrains_aligned[ss][:-lag,:]
        
        train_x = np.concatenate(xtrains_aligned[:s] + xtrains_aligned[s+1:], axis=0)
        train_y = np.concatenate(ytrains_aligned[:s] + ytrains_aligned[s+1:], axis=0)

        test_x = xtrains_aligned[s]
        
        # Time delay embedding    
        if not delay:
            train_x = train_x[:,-1:]
            test_x = test_x[:,-1:]

        # Project to top group PCs for efficient training
        u,sigma,vh = rsvd(train_y, n_components=num_PCs)

        train_y = train_y@vh.T[:,:num_PCs]

        # Preprocessing
        train_x_sc, scale_x = scaler(train_x)
        train_y_sc, scale_y = scaler(train_y)
        
        # Train model
        if nonlinear:
            r_squared, model, encoder, decoder = VAE(train_x_sc, train_y_sc, latent_dim=z_n, beta=beta,
                                                     num_epochs=num_epochs, anneal_step=anneal_step, lr=lr)
        else:
            r_squared, model = linear_regression(train_x_sc, train_y_sc)

        # Test model    
        test_x_sc,_ = scaler(test_x, scale_x)
        
        test_y_hat_sc = model.predict(test_x_sc)
        test_y_hat = scale_y.inverse_transform(test_y_hat_sc)        
        test_y_hat = yscales[s].inverse_transform(test_y_hat@vh[:num_PCs,:]) # project back to brain space
        
        test_y = load_brain(subj,modality)[spacing*(stackmax-1):,:][:-lag,:]

        # Save files

        if delay & nonlinear:
            np.save(outdir + '/' + str(subj) + '_' + modality + '_delay_xtest.npy', test_x_sc)
            np.save(outdir + '/' + str(subj) + '_' + modality + '_ytest.npy', test_y)
            np.save(outdir + '/' + str(subj) + '_' + modality + '_delay_nonlin_ytest_hat.npy', test_y_hat)
            
            encoder.save_weights(outdir + '/' + str(subj) + '_' + modality + '_encoder_weights.h5')
            decoder.save_weights(outdir + '/' + str(subj) + '_' + modality + '_decoder_weights.h5')

        elif delay:
            np.save(outdir + '/' + str(subj) + '_' + modality + '_delay_lin_ytest_hat.npy', test_y_hat)

        elif nonlinear:
            np.save(outdir + '/' + str(subj) + '_' + modality + '_nodelay_nonlin_ytest_hat.npy', test_y_hat)
        
        else:
            np.save(outdir + '/' + str(subj) + '_' + modality + '_nodelay_lin_ytest_hat.npy', test_y_hat)
        
    toc()